In [ ]:
## Setup and Configuration

import sys, json, logging, pickle
from pathlib import Path
import pandas as pd
import numpy as np
from catboost import CatBoostClassifier
from sklearn.linear_model import LogisticRegression

sys.path.insert(0, str(Path.cwd().parent))
from src.feature_extraction import compute_features
from src.utils import load_active_stressors, load_best_combination

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')
log = logging.getLogger(__name__)

# Paths
PROJ_ROOT = Path.cwd().parent
DATA_DIR = PROJ_ROOT / 'data'
MODEL_DIR = DATA_DIR / 'models'

# Load configuration
with open(DATA_DIR / 'best_feature_combination.json', 'r') as f:
    best_combo = json.load(f)['combination'].split('+')

with open(DATA_DIR / 'train_feature_columns.json', 'r') as f:
    train_feature_cols = json.load(f)

with open(DATA_DIR / 'best_thresholds.json', 'r') as f:
    best_thresholds = json.load(f)

active_stressors = load_active_stressors(DATA_DIR)

log.info(f"Loaded {len(best_combo)} feature sets: {best_combo}")
log.info(f"Loaded {len(best_thresholds)} best thresholds")
log.info(f"Loaded {len(active_stressors)} active stressors")

In [ ]:
## Load ENIGMA Isolate Sequences

# Fetch isolate proteins from Spark (cluster environment required)
try:
    from pyspark.sql import SparkSession
    spark = SparkSession.builder.getOrCreate()
    
    # Query isolate from ENIGMA genome depot
    strain = spark.table("enigma_genome_depot_enigma.browser_strain")
    
    # Example: query for a specific isolate (GW821-HT32C06)
    # Modify the filter below to select different isolates
    match = strain.filter("lower(full_name) rlike 'gw821.*ht32c06|ht32c06'")
    
    strain_id = match.select("id").first()[0]
    log.info(f"Found strain ID: {strain_id}")
    
    # Get genome ID
    genome_id = spark.table("enigma_genome_depot_enigma.browser_genome") \
                     .filter(f"strain_id = {strain_id}").select("id").first()[0]
    log.info(f"Found genome ID: {genome_id}")
    
    # Fetch genes and proteins
    gene = spark.table("enigma_genome_depot_enigma.browser_gene")
    protein = spark.table("enigma_genome_depot_enigma.browser_protein")
    
    iso_genes = gene.filter(f"genome_id = {genome_id}").select("protein_id","locus_tag","function")
    iso_prots = protein.join(iso_genes, protein.id == iso_genes.protein_id) \
                       .select(protein.id, protein.sequence, iso_genes.locus_tag, iso_genes.function)
    
    isolate_pd = iso_prots.toPandas()
    log.info(f"Fetched {len(isolate_pd)} proteins from isolate")
    isolate_pd.head()
    
except Exception as e:
    log.warning(f"Could not fetch isolate from Spark: {e}")
    log.info("If running locally without cluster, check for pre-computed features in data/ directory")
    isolate_pd = None

In [ ]:
## Feature Extraction for Isolate

if isolate_pd is not None:
    # Compute features using the same pipeline as training
    iso_feat = compute_features(
        seqs=isolate_pd['sequence'].tolist(),
        organism_name='query_isolate',
        best_combo=best_combo,
        train_columns=train_feature_cols,
        protein_ids=isolate_pd['id'].tolist(),
        esm_cache_path=DATA_DIR / "esm2_embeddings_isolate.parquet",
        device=None  # Will auto-detect GPU if available
    )
    
    log.info(f"Computed features: shape {iso_feat.shape}")
    log.info(f"Feature columns: {iso_feat.shape[1]}")
else:
    log.warning("Cannot compute features without isolate data. Skipping feature extraction.")

In [ ]:
## Load Models and Make Predictions

if isolate_pd is not None:
    predictions = {}
    predictions_proba = {}
    
    for stressor in active_stressors:
        model_path = MODEL_DIR / f"stressor_{stressor}_final.cbm"
        platt_path = MODEL_DIR / f"stressor_{stressor}_platt.pkl"
        
        if not model_path.exists():
            log.warning(f"{stressor}: model not found at {model_path}")
            continue
        
        # Load model
        model = CatBoostClassifier()
        model.load_model(str(model_path))
        
        # Get raw probabilities
        y_proba_raw = model.predict_proba(iso_feat)[:, 1]
        
        # Apply Platt calibration if available
        if platt_path.exists():
            with open(platt_path, 'rb') as f:
                platt = pickle.load(f)
            y_proba = platt.predict_proba(y_proba_raw.reshape(-1, 1))[:, 1]
        else:
            y_proba = y_proba_raw
        
        # Apply threshold-aware prediction
        threshold = best_thresholds.get(stressor, 0.5)
        y_pred = (y_proba >= threshold).astype(int)
        
        predictions[stressor] = y_pred
        predictions_proba[stressor] = y_proba
        
        log.info(f"{stressor}: threshold={threshold:.3f}, n_positive_predictions={y_pred.sum()}")
    
    # Create prediction dataframe
    pred_df = pd.DataFrame(predictions, index=isolate_pd['id'].values)
    pred_proba_df = pd.DataFrame(predictions_proba, index=isolate_pd['id'].values)
    
    # Add metadata
    pred_df['locus_tag'] = isolate_pd['locus_tag'].values
    pred_df['function'] = isolate_pd['function'].values
    
    log.info(f"Generated predictions for {len(pred_df)} proteins across {len(predictions)} stressors")
else:
    log.warning("Cannot make predictions without isolate features")

In [ ]:
## Rank and Output Candidates

if isolate_pd is not None and 'pred_df' in locals():
    # Calculate mean predicted probability across all stressors
    stressor_cols = [col for col in pred_proba_df.columns if col in active_stressors]
    pred_proba_df['mean_stress_probability'] = pred_proba_df[stressor_cols].mean(axis=1)
    
    # Save full predictions
    output_path = DATA_DIR / 'isolate_predictions.csv'
    pred_df.to_csv(output_path)
    log.info(f"Saved predictions to {output_path}")
    
    # Print top candidates per stressor (sorted by probability)
    print("\n" + "="*80)
    print("TOP 10 CANDIDATE PROTEINS PER STRESSOR")
    print("="*80)
    
    for stressor in active_stressors[:15]:  # Show top 15 stressors by frequency
        if stressor not in pred_proba_df.columns:
            continue
        
        # Get top 10 by probability
        top_candidates = pred_proba_df.nlargest(10, stressor)
        
        if len(top_candidates) == 0:
            continue
        
        print(f"\n{stressor.upper()}")
        print("-" * 80)
        print(f"{'Protein ID':<20} {'Locus Tag':<15} {'Probability':<12} {'Prediction':<5} {'Function':<20}")
        print("-" * 80)
        
        for protein_id, row in top_candidates.iterrows():
            prob = row[stressor]
            pred = predictions[stressor][pred_df.index.get_loc(protein_id)] if stressor in predictions else '?'
            locus = pred_df.loc[protein_id, 'locus_tag'] if protein_id in pred_df.index else 'N/A'
            func = pred_df.loc[protein_id, 'function'] if protein_id in pred_df.index else 'N/A'
            
            # Truncate function description
            if isinstance(func, str):
                func = func[:20]
            else:
                func = 'N/A'
            
            print(f"{str(protein_id):<20} {str(locus):<15} {prob:<12.4f} {str(pred):<5} {str(func):<20}")
    
    # Overall highest-risk proteins (across all stressors)
    print("\n" + "="*80)
    print("TOP 10 HIGHEST-RISK PROTEINS (BY MEAN PROBABILITY)")
    print("="*80)
    top_overall = pred_proba_df.nlargest(10, 'mean_stress_probability')
    print(f"\n{'Protein ID':<20} {'Locus Tag':<15} {'Mean Probability':<18} {'Function':<30}")
    print("-" * 80)
    
    for protein_id, row in top_overall.iterrows():
        locus = pred_df.loc[protein_id, 'locus_tag'] if protein_id in pred_df.index else 'N/A'
        func = pred_df.loc[protein_id, 'function'] if protein_id in pred_df.index else 'N/A'
        
        if isinstance(func, str):
            func = func[:30]
        else:
            func = 'N/A'
        
        print(f"{str(protein_id):<20} {str(locus):<15} {row['mean_stress_probability']:<18.4f} {str(func):<30}")
else:
    log.warning("Cannot output predictions without data")